In [22]:
import numpy as np
import pandas as pd
from evabox import utils, ephys, plotting
from pathlib import Path

In [ ]:
animal = "eb08"
date = "20260425"
params = utils.params_dict(animal, date)

{'animal': 'eb08', 'date': '20260425', 'merged': True, 'subsession': None, 'fps': 120, 'fs': 30000, 'ttl_type': 'continuous', 'data_root': PosixPath('/storage3/eva/data')}


In [3]:
params

{'animal': 'eb08',
 'date': '20260425',
 'merged': True,
 'subsession': None,
 'fps': 120,
 'fs': 30000,
 'ttl_type': 'continuous',
 'data_root': PosixPath('/storage3/eva/data')}

In [5]:
ttl_type = params['ttl_type']
paths = utils.build_paths(params['animal'], params['date'])

In [6]:
paths

{'data_raw_ephys': PosixPath('/storage3/eva/data/raw/eb08/eb08_20260425/ephys'),
 'data_raw_motive': PosixPath('/storage3/eva/data/raw/eb08/eb08_20260425/motive'),
 'data_processed_ks': PosixPath('/storage3/eva/data/processed/eb08/eb08_20260425/ephys/eb08_20260425_merged/kilosort4'),
 'data_processed_rec': PosixPath('/storage3/eva/data/processed/eb08/eb08_20260425/ephys/eb08_20260425_merged/recording/raw_merged'),
 'data_processed_ttl': PosixPath('/storage3/eva/data/processed/eb08/eb08_20260425/ephys/eb08_20260425_merged/recording/ttl'),
 'data_processed_unitmatch': PosixPath('/storage3/eva/data/processed/eb08/unitmatch'),
 'data_processed_motive': PosixPath('/storage3/eva/data/processed/eb08/eb08_20260425/motive'),
 'results_animal': PosixPath('/storage3/eva/results/eb08'),
 'results_session': PosixPath('/storage3/eva/results/eb08/eb08_20260425')}

In [7]:
ttl_on = np.load(paths['data_processed_ttl'] / "ttl_merged_onsets.npy")
ttl_off = np.load(paths['data_processed_ttl'] / "ttl_merged_offsets.npy")

In [15]:
len(ttl_on) == len(ttl_off)

True

In [16]:
def behavioral_periods(ttl_path, ttl_type, fs, fps, frame_jitter=0.01, verbose=True):
    """Detect start and end samples of behavioral periods from TTL pulses.
    Input trigger = Rising Edge --> start and stop is detected by rising edge of ttl!

    Two modes:
      - "continuous": camera fires a TTL on every frame; periods are inferred
        from gaps larger than one frame interval (plus jitter tolerance).
      - "start-end": one rising edge marks the start of a period, the next
        marks the end.

    Args:
        ttl_path (Path): folder with `ttl_merged_onsets.npy` and
            `ttl_merged_offsets.npy` (sample indices of rising/falling edges).
        ttl_type (str): "continuous" or "start-end".
        fs (float): ephys sampling rate in Hz.
        fps (float): camera frame rate in Hz (only used for "continuous").
        frame_jitter (float): tolerance above one frame interval before a gap
            counts as a period boundary. Defaults to 0.01.
        verbose (bool): print period durations in minutes. Defaults to True.

    Returns:
        (periods_start, periods_end): two np.ndarray of sample indices,
        one entry per behavioral period.
    """
    ttl_on = np.load(ttl_path / "ttl_merged_onsets.npy")
    ttl_off = np.load(ttl_path / "ttl_merged_offsets.npy")
    assert len(ttl_on) == len(ttl_off), f"TTL mismatch: {len(ttl_on)} on vs {len(ttl_off)} off"

    if ttl_type == "continuous":
        diffs = np.diff(ttl_on)
        gap_threshold = fs / fps * (1 + frame_jitter)  # one frame + jitter, in samples
        gap_indices = np.where(diffs > gap_threshold)[0]
        periods_start = np.concatenate([[ttl_on[0]], ttl_on[gap_indices + 1]])
        periods_end = np.concatenate([ttl_on[gap_indices], [ttl_on[-1]]])
    elif ttl_type == "start-end":
        assert len(ttl_on) % 2 == 0, f"start-end needs even number of TTLs, got {len(ttl_on)}"
        periods_start = ttl_on[::2]
        periods_end = ttl_on[1::2]
    else:
        raise ValueError(f"Unknown ttl_type {ttl_type!r}; expected 'continuous' or 'start-end'")

    if verbose:
        durations_min = (periods_end - periods_start) / fs / 60
        print(f"{len(periods_start)} periods, durations (min): {durations_min}")

    return periods_start, periods_end

In [20]:
periods_start, periods_end = behavioral_periods(paths['data_processed_ttl'], params["ttl_type"], params["fs"], params["fps"])

7 periods, durations (min): [41.03095167 37.31715778 60.74098889 30.38801944 43.41613944 38.54655611
 46.76563833]


In [ ]:
import yaml
data_root=Path("/storage3/eva/data")

with open(data_root / "meta" / "meta.yaml") as f:
    meta = yaml.safe_load(f)

In [27]:
meta

{'eb08': {'url': 'https://docs.google.com/spreadsheets/d/e/2PACX-1vTWu-h4pjar9BY-hicinzN0hKtCN5KlgIaRSI_MOCuyrie3F9xOe9QBL53dRJ7TDYfeT2TIqny8EyKM/pub?output=csv',
  'tabs': {'Surgery': '383869021',
   'SubSessions': '0',
   'Sessions': '1993615845'}},
 'eb02': {'url': '...',
  'tabs': {'Surgery': '...', 'SubSessions': '...', 'Sessions': '...'}}}

In [33]:
def load_tab(meta, animal, tab):
    base = meta[animal]["url"].replace("?output=csv", "")
    gid = meta[animal]["tabs"][tab]
    return pd.read_csv(f"{base}?gid={gid}&single=true&output=csv")

In [29]:
tab = "SubSessions"
SubSessions_csv = load_tab(animal, tab)

In [32]:
SubSessions_csv.head()

,SubSession,Task,LEDcsv,LEDavi,Exposure,Threshold,RigidBodies,PelletRate,Comments
0,eb08_20260423_1,sleep,DC,DC,3000.0,120.0,NaN,NaN,NaN
1,eb08_20260423_2,"foraging, BOF",NaN,NaN,NaN,NaN,Rat_NP,NaN,NaN
2,eb08_20260423_3,sleep,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,eb08_20260423_4,"foraging, COF",NaN,NaN,NaN,NaN,Rat_NP,NaN,OpenEpys stoped responding at the end but stil...
4,eb08_20260423_5,sleep,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
def get_subsession_names(animal, date, data_root=Path("/storage3/eva/data")):
    with open(data_root / "meta" / "meta.yaml") as f:
        meta = yaml.safe_load(f)

    tab = "SubSessions"
    SubSessions_csv = load_tab(meta, animal, tab)


    prefix = f"{animal}_{date}_"
    mask = SubSessions_csv["SubSession"].str.startswith(prefix)
    df_filtered = SubSessions_csv[mask].copy()
    df_filtered["num"] = df_filtered["SubSession"].str.extract(r"_(\d+)$")
    df_filtered["task_clean"] = (
        df_filtered["Task"]
        .str.strip()
        .str.replace(",", "", regex=False)
        .str.replace(r"\s+", "_", regex=True)
    )
    return (df_filtered["task_clean"] + "_" + df_filtered["num"]).tolist()

In [35]:
names = get_subsession_names(animal, date)
names

['sleep_1',
 'foraging_BOF_2',
 'sleep_3',
 'foraging_COF_4',
 'sleep_5',
 'foraging_BOF_6',
 'sleep_7']

In [37]:
periods = pd.DataFrame(
    {
        "name": names,
        "start_sample": periods_start,
        "stop_sample": periods_end,
    }
)
periods["start_time_s"] = periods["start_sample"] / params["fs"]
periods["stop_time_s"] = periods["stop_sample"] / params["fs"]
periods["duration_min"] = (periods["stop_sample"] - periods["start_sample"]) / params["fs"] / 60
periods

,name,start_sample,stop_sample,start_time_s,stop_time_s,duration_min
0,sleep_1,171276,74026989,5.709200,2467.566300,41.030952
1,foraging_BOF_2,74407852,141578736,2480.261733,4719.291200,37.317158
2,sleep_3,142317768,251651548,4743.925600,8388.384933,60.740989
3,foraging_COF_4,251980965,306679400,8399.365500,10222.646667,30.388019
4,sleep_5,307002768,385151819,10233.425600,12838.393967,43.416139
5,foraging_BOF_6,385447154,454830955,12848.238467,15161.031833,38.546556
6,sleep_7,455107666,539285815,15170.255533,17976.193833,46.765638
